In [1]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_groq import ChatGroq


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

In [9]:
model=llm

In [2]:
api_key = os.getenv("TAVILY_API_KEY")
api_key

'tvly-nagmN-36bgOd1HjzUOhzNkKL0nHXB67s44WczkKxOM58cjXA'

In [3]:
from tavily import TavilyClient
# To install: pip install tavily-python
from tavily import TavilyClient
client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)

api_key = os.getenv("TAVILY_API_KEY")
if not api_key:
    raise ValueError("Set TAVILY_API_KEY to a valid Tavily API key.")

client = TavilyClient(api_key=api_key)
response = client.search(
    query="What nation hosted the Euro 2024?",
    search_depth="advanced",
    include_favicon=True,
    include_usage=True
)
print(response)

{'query': 'What nation hosted the Euro 2024?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.uefa.com/news/0257-0e13b161b2e8-4a3fd5615e0c-1000--euro-2024-the-lowdown', 'title': 'EURO 2024: All you need to know | UEFA.com', 'score': 0.9245858, 'content': "## Article body\n\n## Where was EURO 2024 held?\n\nGermany hosted EURO 2024, having been chosen to stage the 17th edition of the UEFA European Championship at a UEFA Executive Committee meeting in Nyon on 27 September 2018. Türkiye was the only other nation that bid to hold UEFA EURO 2024. West Germany hosted the 1988 edition, but this was the first time Germany staged the UEFA European Championship since reunification. The 2006 FIFA World Cup also took place in the country.\n\n## What were the dates for EURO 2024?\n\nThe UEFA EURO 2024 final tournament began on 14 June and ran until 14 July 2024. The group stage concluded on 26 June, while the knockout stage began on 29 June. [...] As the 

In [11]:
import os
from tavily import TavilyClient
from langchain.tools import tool

client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)

@tool
def tavily_search(query: str) -> str:
    """Search the web using Tavily and return relevant results."""

    response = client.search(
        query=query,
        search_depth="advanced",
        max_results=5,
        include_favicon=True
    )

    return str(response["results"])

In [12]:
llm_with_tools = model.bind_tools([tavily_search])

In [19]:
response = llm_with_tools.invoke(
    "Search the web: Which nation hosted Euro 2024?"
)


In [21]:
print(response)

content='' additional_kwargs={'reasoning_content': 'The user asks: "Search the web: Which nation hosted Euro 2024?" We need to search web. Use function tavily_search.', 'tool_calls': [{'id': 'fc_31d04167-45f1-49b0-ad5d-9bf2f19db746', 'function': {'arguments': '{"query":"Euro 2024 host nation"}', 'name': 'tavily_search'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 137, 'total_tokens': 201, 'completion_time': 0.134516766, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.007061091, 'prompt_tokens_details': None, 'queue_time': 0.370182677, 'total_time': 0.141577857}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1ca38-7588-76d1-a9b3-0b68cf8e240b-0' tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Euro 2024 host nation'}, 'id': 'fc_31d04167-45f1-49b0-ad5d-9bf

In [ ]:
from langchain_core.messages import ToolMessage

response = llm_with_tools.invoke(
    "Search the web: Which nation hosted Euro 2024?"
)

# Print LLM tool call
response.pretty_print()

# Execute Tavily tool
for tool_call in response.tool_calls:
    result = tavily_search.invoke(tool_call["args"])

    tool_message = ToolMessage(
        content=result,
        tool_call_id=tool_call["id"]
    )

    tool_message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  tavily_search (fc_83e0eb35-5db3-490a-8abc-706c798a8aac)
 Call ID: fc_83e0eb35-5db3-490a-8abc-706c798a8aac
  Args:
    query: Euro 2024 host nation
================================= Tool Message =================================

[{'url': 'https://en.wikipedia.org/wiki/UEFA_Euro_2024', 'title': 'UEFA Euro 2024 - Wikipedia', 'content': '| Fußball-Europameisterschaft 2024 (in German) |\n| United by Football. Vereint im Herzen Europas. (United in the heart of Europe.) |\n| Tournament details |\n| Host country | Germany |\n| Dates | 14 June – 14 July |\n| Teams | 24 |\n| Venues | 10 (in 10 host cities) |\n| Final positions |\n| Champions | Spain (4th title) |\n| Runners-up | England |\n| Third place | France  Netherlands |\n| Tournament statistics |\n| Matches played | 51 |\n| Goals scored | 117 (2.29 per match) |\n| Attendance | 2,681,288 (52,574 per match) |\n| Top scorers |  England Harry Kane 

: 

In [5]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

In [ ]:
from langchain.agents import create_agent


In [10]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent


# Initialize Tavily Search Tool
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
)

agent = create_agent(model, [tavily_search_tool])

user_input = "What nation hosted the Euro 2024? Include only wikipedia sources."

stream = agent.stream_events({"messages": user_input}, version="v3")
for snapshot in stream.values:
    snapshot["messages"][-1].pretty_print()

c:\Agentic AI\AI-Travel-Planning-System-using-LangGraph\travel_agent\Lib\site-packages\langgraph\pregel\main.py:3804: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
c:\Agentic AI\AI-Travel-Planning-System-using-LangGraph\travel_agent\Lib\site-packages\langgraph\pregel\main.py:3646: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


================================ Human Message =================================

What nation hosted the Euro 2024? Include only wikipedia sources.
================================== Ai Message ==================================

[{'type': 'reasoning', 'reasoning': 'The user asks: "What nation hosted the Euro 2024? Include only wikipedia sources."\n\nWe need to answer: The nation that hosted Euro 2024 is Germany. Provide answer with citation from Wikipedia. Must include only Wikipedia sources. So we need to search Wikipedia page for UEFA Euro 2024. Use search tool with query "UEFA Euro 2024 Wikipedia".', 'index': 0}, {'type': 'tool_call', 'name': 'tavily_search', 'args': {'include_domains': ['en.wikipedia.org'], 'query': 'UEFA Euro 2024 Wikipedia', 'search_depth': 'fast'}, 'id': 'fc_e98475b2-50f3-4653-a060-1999119f3cc5'}]
Tool Calls:
  tavily_search (fc_e98475b2-50f3-4653-a060-1999119f3cc5)
 Call ID: fc_e98475b2-50f3-4653-a060-1999119f3cc5
  Args:
    include_domains: ['en.wikipedia.or